# Avtech Pulse Generator Test Notebook

Instantiates `instruments.avtech.Avtech`, connects, reads back its status,
sets pulse parameters (width, voltage, frequency, trigger mode), and
demonstrates the `ramp_to_voltage` safety helper.

**No real instrument is connected to this development machine.** Every
live-hardware call below is wrapped in `try`/`except` so this notebook
runs cleanly end-to-end and prints a clear "hardware not available"
message instead of raising, when run here. On the lab computer (with the
Avtech actually connected), the same cells should run against real
hardware.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root (containing `instruments/`) is importable,
# whether this notebook is run from the repo root or from within notebooks/.
project_root = Path.cwd()
if not (project_root / "instruments").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
from instruments.avtech import Avtech, AvtechError
from instruments import config

print(f"Default Avtech VISA address: {config.AVTECH_VISA_ADDRESS}")

## Connect and enter remote mode

In [ ]:
avtech = Avtech()
avtech_connected = False

try:
    avtech.connect()
    avtech.remote()
    avtech_connected = True
    print(f"Connected to Avtech at {avtech.resource}")
    print("IDN:", avtech.idn())
except Exception as exc:
    print(f"Hardware not available: could not connect to Avtech ({exc})")

## Status check

In [ ]:
try:
    if not avtech_connected:
        raise RuntimeError("Avtech not connected")
    print("Pulse width (s):", avtech.get_pulse_width(), "range:", avtech.get_pulse_width_range())
    print("Pulse voltage (V):", avtech.get_voltage(), "range:", avtech.get_voltage_range())
    print("Pulse frequency (Hz):", avtech.get_frequency(), "range:", avtech.get_frequency_range())
    print("Pulse shape:", avtech.get_shape())
    print("Trigger mode:", avtech.get_trigger())
    print("Gate mode:", avtech.get_gate())
    avtech.check_error()
    print("No pending Avtech error.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Set pulse parameters

In [ ]:
try:
    if not avtech_connected:
        raise RuntimeError("Avtech not connected")
    avtech.set_pulse_width(80e-6)
    avtech.set_voltage(5.0)
    avtech.set_frequency(1.0)
    avtech.set_trigger("EXT")
    print("Pulse width, voltage, frequency, and trigger mode set.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Output enable/disable

In [ ]:
try:
    if not avtech_connected:
        raise RuntimeError("Avtech not connected")
    avtech.output_on()
    print("Output state:", avtech.get_output())
    avtech.output_off()
    print("Output turned back off.")
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Ramp to voltage (safety helper)

`ramp_to_voltage` steps the pulse amplitude gradually instead of jumping
directly to the target, to avoid stressing the device under test. A short
`sleep_time` is used here only to keep this demo fast; the lab default
(2 s) is more conservative.

In [ ]:
try:
    if not avtech_connected:
        raise RuntimeError("Avtech not connected")
    avtech.ramp_to_voltage(5.0, step_size=1.0, sleep_time=0.1)
    print("Ramped pulse voltage to 5.0 V. Current voltage:", avtech.get_voltage())
except Exception as exc:
    print(f"Hardware not available or command failed: {exc}")

## Return to local control and disconnect

In [ ]:
try:
    if avtech_connected:
        avtech.local()
        avtech.disconnect()
        print("Returned to local control and disconnected from Avtech.")
    else:
        print("Skipping disconnect: Avtech was never connected.")
except Exception as exc:
    print(f"Error during disconnect: {exc}")

## Context manager usage

In [ ]:
try:
    with Avtech() as avr:
        print("IDN:", avr.idn())
except Exception as exc:
    print(f"Hardware not available: {exc}")